# 🧠 Machine Learning - Prédiction LSTM

Ce notebook va entraîner un modèle **LSTM (Long Short-Term Memory)** pour prédire les mouvements futurs des indices synthétiques.

## 🎯 Objectif

Prédire le **prix futur** basé sur les prix passés, en utilisant :
- Les 60 dernières bougies (séquence)
- Un réseau LSTM à 2 couches
- Prédiction du prochain prix

## ⚙️ Instructions

1. **Exécuter les cellules dans l'ordre**
2. **Modifier le SYMBOL** dans la cellule 2
3. **Attendre l'entraînement** (~5-15 min selon ton CPU/GPU)
4. **Analyser les résultats**

In [ ]:
# ========================================
# CELLULE 1 : IMPORTS
# ========================================

import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# PyTorch (si installé)
try:
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader
    from sklearn.preprocessing import MinMaxScaler
    TORCH_AVAILABLE = True
    print(f"✅ PyTorch version: {torch.__version__}")
    print(f"✅ CUDA available: {torch.cuda.is_available()}")
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"✅ Using device: {device}")
except ImportError:
    print("❌ PyTorch not installed. Installing...")
    print("Run: pip install torch")
    TORCH_AVAILABLE = False

import warnings
warnings.filterwarnings('ignore')

if TORCH_AVAILABLE:
    print("\n✅ Ready to train LSTM!")
else:
    print("\n⚠️ Install PyTorch first: pip install torch")

In [ ]:
# ========================================
# CELLULE 2 : CONFIGURATION
# ========================================

# ⚠️ MODIFIER ICI
SYMBOL = "Volatility_100_Index"  # Ou "Crash_500_Index", "Boom_500_Index", etc.
TIMEFRAME = "M15"

# Hyperparamètres
SEQUENCE_LENGTH = 60  # Utiliser 60 bougies pour prédire la suivante
PREDICTION_HORIZON = 1  # Prédire 1 bougie dans le futur
HIDDEN_SIZE = 128  # Taille de la couche cachée LSTM
NUM_LAYERS = 2  # Nombre de couches LSTM
LEARNING_RATE = 0.001
EPOCHS = 50  # Nombre d'epochs (augmenter pour meilleure performance)
BATCH_SIZE = 32

# Split ratios
TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
TEST_RATIO = 0.15

print(f"Configuration:")
print(f"  Symbol: {SYMBOL}")
print(f"  Sequence length: {SEQUENCE_LENGTH}")
print(f"  Hidden size: {HIDDEN_SIZE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Device: {device}")

In [ ]:
# ========================================
# CELLULE 3 : CHARGEMENT DES DONNÉES
# ========================================

import os

data_path = f"../data/raw/{SYMBOL}_{TIMEFRAME}.parquet"

print(f"Loading {data_path}...")

if not os.path.exists(data_path):
    print(f"❌ File not found: {data_path}")
    print("\nAvailable files:")
    for f in os.listdir("../data/raw"):
        if f.endswith('.parquet'):
            print(f"  - {f}")
else:
    df = pd.read_parquet(data_path)
    
    print(f"✅ Data loaded: {len(df)} rows")
    print(f"📅 Period: {df.index[0]} to {df.index[-1]}")
    
    # Utiliser seulement le prix de clôture
    prices = df['Close'].values.reshape(-1, 1)
    
    print(f"\n📊 Price statistics:")
    print(f"  Min:  {prices.min():.2f}")
    print(f"  Max:  {prices.max():.2f}")
    print(f"  Mean: {prices.mean():.2f}")
    print(f"  Std:  {prices.std():.2f}")

In [ ]:
# ========================================
# CELLULE 4 : PRÉPARATION DES DONNÉES
# ========================================

print("Preparing data for LSTM...\n")

# 1. Normalisation (important pour les réseaux de neurones)
scaler = MinMaxScaler(feature_range=(0, 1))
prices_scaled = scaler.fit_transform(prices)

print(f"✅ Data normalized to [0, 1]")

# 2. Créer les séquences
def create_sequences(data, seq_length, pred_horizon):
    X, y = [], []
    for i in range(len(data) - seq_length - pred_horizon + 1):
        X.append(data[i:i + seq_length])
        y.append(data[i + seq_length])
    return np.array(X), np.array(y)

X, y = create_sequences(prices_scaled, SEQUENCE_LENGTH, PREDICTION_HORIZON)

print(f"✅ Sequences created")
print(f"  X shape: {X.shape} (samples, sequence_length, features)")
print(f"  y shape: {y.shape} (samples, features)")

# 3. Split train/val/test
n_samples = len(X)
train_size = int(n_samples * TRAIN_RATIO)
val_size = int(n_samples * VAL_RATIO)

X_train = X[:train_size]
y_train = y[:train_size]

X_val = X[train_size:train_size + val_size]
y_val = y[train_size:train_size + val_size]

X_test = X[train_size + val_size:]
y_test = y[train_size + val_size:]

print(f"\n✅ Data split:")
print(f"  Train: {len(X_train)} samples ({TRAIN_RATIO*100:.0f}%)")
print(f"  Val:   {len(X_val)} samples ({VAL_RATIO*100:.0f}%)")
print(f"  Test:  {len(X_test)} samples ({TEST_RATIO*100:.0f}%)")

# 4. Convertir en tenseurs PyTorch
X_train_tensor = torch.FloatTensor(X_train)
y_train_tensor = torch.FloatTensor(y_train)

X_val_tensor = torch.FloatTensor(X_val)
y_val_tensor = torch.FloatTensor(y_val)

X_test_tensor = torch.FloatTensor(X_test)
y_test_tensor = torch.FloatTensor(y_test)

# 5. Créer les DataLoaders
train_dataset = torch.utils.data.TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

val_dataset = torch.utils.data.TensorDataset(X_val_tensor, y_val_tensor)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

test_dataset = torch.utils.data.TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"\n✅ DataLoaders created")

In [ ]:
# ========================================
# CELLULE 5 : DÉFINITION DU MODÈLE LSTM
# ========================================

class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size, dropout=0.2):
        super(LSTMModel, self).__init__()
        
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # LSTM layers
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0,
            batch_first=True
        )
        
        # Fully connected layers
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, output_size)
        )
    
    def forward(self, x):
        # x shape: (batch_size, sequence_length, input_size)
        
        # LSTM forward
        lstm_out, _ = self.lstm(x)
        
        # Use last output
        last_output = lstm_out[:, -1, :]
        
        # Fully connected
        output = self.fc(last_output)
        
        return output

# Créer le modèle
model = LSTMModel(
    input_size=1,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    output_size=1
).to(device)

# Compter les paramètres
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"✅ Model created:")
print(model)
print(f"\n📊 Total trainable parameters: {num_params:,}")

In [ ]:
# ========================================
# CELLULE 6 : ENTRAÎNEMENT
# ========================================

print(f"🚀 Starting training for {EPOCHS} epochs...\n")

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Historique
train_losses = []
val_losses = []

# Early stopping
best_val_loss = float('inf')
patience = 10
patience_counter = 0

for epoch in range(EPOCHS):
    # Training
    model.train()
    train_loss = 0
    
    for batch_x, batch_y in train_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        
        # Forward
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
    
    train_loss /= len(train_loader)
    
    # Validation
    model.eval()
    val_loss = 0
    
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            
            val_loss += loss.item()
    
    val_loss /= len(val_loader)
    
    # Save history
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    
    # Print progress
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS} - Train Loss: {train_loss:.6f}, Val Loss: {val_loss:.6f}")
    
    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        # Save best model
        torch.save(model.state_dict(), f'../data/models/lstm_{SYMBOL}_{TIMEFRAME}_best.pth')
    else:
        patience_counter += 1
    
    if patience_counter >= patience:
        print(f"\n⚠️ Early stopping at epoch {epoch+1}")
        break

print(f"\n✅ Training complete!")
print(f"📊 Best validation loss: {best_val_loss:.6f}")

# Load best model
model.load_state_dict(torch.load(f'../data/models/lstm_{SYMBOL}_{TIMEFRAME}_best.pth'))
print(f"✅ Best model loaded")

In [ ]:
# ========================================
# CELLULE 7 : VISUALISATION DE L'ENTRAÎNEMENT
# ========================================

plt.figure(figsize=(12, 5))
plt.plot(train_losses, label='Train Loss', alpha=0.8)
plt.plot(val_losses, label='Val Loss', alpha=0.8)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Training History')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Final Train Loss: {train_losses[-1]:.6f}")
print(f"Final Val Loss: {val_losses[-1]:.6f}")

In [ ]:
# ========================================
# CELLULE 8 : ÉVALUATION SUR TEST SET
# ========================================

print("Evaluating on test set...\n")

model.eval()
predictions = []
actuals = []

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x = batch_x.to(device)
        
        outputs = model(batch_x)
        
        predictions.extend(outputs.cpu().numpy())
        actuals.extend(batch_y.numpy())

predictions = np.array(predictions)
actuals = np.array(actuals)

# Inverse transform pour avoir les vrais prix
predictions_real = scaler.inverse_transform(predictions)
actuals_real = scaler.inverse_transform(actuals)

# Métriques
mse = np.mean((predictions_real - actuals_real) ** 2)
rmse = np.sqrt(mse)
mae = np.mean(np.abs(predictions_real - actuals_real))
mape = np.mean(np.abs((actuals_real - predictions_real) / actuals_real)) * 100

# Directional accuracy
direction_pred = np.diff(predictions_real.flatten()) > 0
direction_actual = np.diff(actuals_real.flatten()) > 0
directional_accuracy = np.mean(direction_pred == direction_actual) * 100

print("=" * 60)
print("TEST SET RESULTS")
print("=" * 60)
print(f"\n📊 Error Metrics:")
print(f"  RMSE: {rmse:.4f}")
print(f"  MAE:  {mae:.4f}")
print(f"  MAPE: {mape:.2f}%")
print(f"\n🎯 Directional Accuracy: {directional_accuracy:.2f}%")
print(f"   (Predicts correct direction {directional_accuracy:.1f}% of the time)")

if directional_accuracy > 55:
    print(f"\n✅ GOOD! Model beats random (50%)")
else:
    print(f"\n⚠️ WARNING: Model not better than random")

print("\n" + "=" * 60)

In [ ]:
# ========================================
# CELLULE 9 : VISUALISATION DES PRÉDICTIONS
# ========================================

# Prendre les 200 dernières prédictions
n_display = min(200, len(predictions_real))

fig = go.Figure()

# Valeurs réelles
fig.add_trace(go.Scatter(
    y=actuals_real[-n_display:].flatten(),
    mode='lines',
    name='Actual',
    line=dict(color='blue', width=2)
))

# Prédictions
fig.add_trace(go.Scatter(
    y=predictions_real[-n_display:].flatten(),
    mode='lines',
    name='Predicted',
    line=dict(color='red', width=2, dash='dash')
))

fig.update_layout(
    title=f'LSTM Predictions vs Actual - {SYMBOL}',
    xaxis_title='Time Step',
    yaxis_title='Price',
    height=600,
    hovermode='x unified'
)

fig.show()

print(f"Showing last {n_display} predictions")

In [ ]:
# ========================================
# CELLULE 10 : SCATTER PLOT
# ========================================

plt.figure(figsize=(10, 10))
plt.scatter(actuals_real, predictions_real, alpha=0.3, s=10)
plt.plot([actuals_real.min(), actuals_real.max()], 
         [actuals_real.min(), actuals_real.max()], 
         'r--', lw=2, label='Perfect Prediction')
plt.xlabel('Actual Price')
plt.ylabel('Predicted Price')
plt.title('Predictions vs Actual')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("Points closer to the red line = better predictions")

In [ ]:
# ========================================
# CELLULE 11 : PRÉDIRE LE FUTUR
# ========================================

print("🔮 Predicting the next price...\n")

# Prendre les 60 dernières bougies
last_sequence = prices_scaled[-SEQUENCE_LENGTH:]

# Convertir en tensor
last_sequence_tensor = torch.FloatTensor(last_sequence).unsqueeze(0).to(device)

# Prédire
model.eval()
with torch.no_grad():
    prediction_scaled = model(last_sequence_tensor)
    prediction_real = scaler.inverse_transform(prediction_scaled.cpu().numpy())

current_price = prices[-1][0]
predicted_price = prediction_real[0][0]
change = predicted_price - current_price
change_pct = (change / current_price) * 100

print("=" * 60)
print("NEXT PRICE PREDICTION")
print("=" * 60)
print(f"\n📊 Current Price:   {current_price:.2f}")
print(f"🔮 Predicted Price: {predicted_price:.2f}")
print(f"📈 Change:          {change:+.2f} ({change_pct:+.2f}%)")

if change > 0:
    print(f"\n✅ BULLISH - Model predicts price will GO UP")
else:
    print(f"\n⚠️ BEARISH - Model predicts price will GO DOWN")

print("\n" + "=" * 60)
print("\n⚠️ DISCLAIMER:")
print("   - This is a prediction, not financial advice")
print("   - Always use stop-loss")
print("   - Past performance doesn't guarantee future results")
print("=" * 60)

In [ ]:
# ========================================
# CELLULE 12 : RÉSUMÉ FINAL
# ========================================

print("\n" + "=" * 70)
print(" " * 25 + "🧠 LSTM MODEL SUMMARY 🧠")
print("=" * 70)

print(f"\n📊 DATASET:")
print(f"   Symbol:           {SYMBOL}")
print(f"   Timeframe:        {TIMEFRAME}")
print(f"   Total samples:    {len(df)}")
print(f"   Training samples: {len(X_train)}")
print(f"   Test samples:     {len(X_test)}")

print(f"\n🧠 MODEL:")
print(f"   Architecture:     LSTM")
print(f"   Hidden size:      {HIDDEN_SIZE}")
print(f"   Num layers:       {NUM_LAYERS}")
print(f"   Parameters:       {num_params:,}")
print(f"   Sequence length:  {SEQUENCE_LENGTH}")

print(f"\n📈 PERFORMANCE:")
print(f"   RMSE:             {rmse:.4f}")
print(f"   MAE:              {mae:.4f}")
print(f"   Directional Acc:  {directional_accuracy:.2f}%")

print(f"\n🔮 NEXT PREDICTION:")
print(f"   Current:          {current_price:.2f}")
print(f"   Predicted:        {predicted_price:.2f} ({change_pct:+.2f}%)")

if directional_accuracy > 55:
    print(f"\n✅ MODEL STATUS: GOOD - Beats random walk!")
    print(f"   The model can predict direction {directional_accuracy:.1f}% of the time")
    print(f"   This is {directional_accuracy - 50:.1f}% better than random!")
else:
    print(f"\n⚠️ MODEL STATUS: NEEDS IMPROVEMENT")
    print(f"   Try:")
    print(f"   - More epochs (increase EPOCHS)")
    print(f"   - More data (extract more days)")
    print(f"   - Different hyperparameters")
    print(f"   - Add more features (volume, indicators, etc.)")

print("\n" + "=" * 70)
print("\n🎯 NEXT STEPS:")
print("   1. Try different symbols (Crash 500, Boom 500, etc.)")
print("   2. Experiment with hyperparameters")
print("   3. Add technical indicators as features")
print("   4. Create a trading strategy based on predictions")
print("   5. Backtest the strategy")
print("\n" + "=" * 70)